Завантаження бібліотек

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

1. Завантаження датасету з бібліотеки seaborn

In [2]:
df=sns.load_dataset('titanic')

2. Перегляд перших рядків датасету

In [3]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


3. Перегляд інформації про датасет

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


4. Статистика по всім стовпцям

In [5]:
df.describe(include='all')

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
count,891.000000,891.000000,891,714.000000,891.000000,891.000000,891.000000,889,891,891,891,203,889,891,891
unique,NaN,NaN,2,NaN,NaN,NaN,NaN,3,3,3,2,7,3,2,2
top,NaN,NaN,male,NaN,NaN,NaN,NaN,S,Third,man,True,C,Southampton,no,True
freq,NaN,NaN,577,NaN,NaN,NaN,NaN,644,491,537,537,59,644,549,537
mean,0.383838,2.308642,NaN,29.699118,0.523008,0.381594,32.204208,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,0.486592,0.836071,NaN,14.526497,1.102743,0.806057,49.693429,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,0.000000,1.000000,NaN,0.420000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,0.000000,2.000000,NaN,20.125000,0.000000,0.000000,7.910400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,0.000000,3.000000,NaN,28.000000,0.000000,0.000000,14.454200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,1.000000,3.000000,NaN,38.000000,1.000000,0.000000,31.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


5. Створення єдиного стовпчика relatives, що вказує кількість родичів для кожного пасажира на борту, замість:
- кількість братів/сестер або чоловіків/дружин на борту;
- кількість батьків або дітей на борту;
          
Результат представлений у вигляді підрахунку кількості пропущених значень (NaN) у стовпці у новому стовпці.

In [6]:
def relatives(df):
    df["relatives"] = df['sibsp'] + df['parch']
    df.loc[df['alone'] == True, 'relatives'] = 0
    df.drop(columns=['sibsp', 'parch', 'alone'], inplace=True)

    return df['relatives'].isna().sum()

relatives(df)

np.int64(0)

6. Розрахунок частоти розподілу кількості родичів через групування та агрегацію

In [7]:
df.groupby('relatives')['relatives'].count().reset_index(name='count')

,relatives,count
0,0,537
1,1,161
2,2,102
3,3,29
4,4,15
5,5,22
6,6,12
7,7,6
8,10,7


7. Використовуючи цикл заміни кількість родичів, що перевищує число 5(п'ять) на значення "above 5"(понад п'ять).       
Запиши значення в новий стовпчик ʼrelatives_categoryʼ.     
Результат представ у вигляді таблиці, побудованої з використанням групування та агрегації:

In [8]:
df["relatives_category"] = ""

for i in range(len(df)):
    if df.loc[i, "relatives"] > 5:
        df.loc[i, "relatives_category"] = "above 5"
    else:
        df.loc[i, "relatives_category"] = df.loc[i, "relatives"]

result = df.groupby("relatives_category")["relatives_category"].count().reset_index(name="count")
print(result)

  relatives_category  count
0                  0    537
1                  1    161
2                  2    102
3                  3     29
4                  4     15
5                  5     22
6            above 5     25


8. Необхідно вивести на екран статистику по відсотковому показнику пасажирів з кількістю родичів більше 5 відносно загального числа пасажирів для кожного з міст посадки. Для цього:
- порахуй загальну кількість пасажирів в кожному з міст посадки,
- знайди число пасажирів з кількістю родичів більше 5 в кожному з міст посадки,
- поділи ці два стовчики між собою, перетворивши результат у відсотки (ціле число).        
Відобрази статистику порахованих показників, згруповану по містах посадки.

In [9]:
result = df.groupby('embark_town').agg(
    total=('embark_town', 'count'),
    above_5=('relatives_category', lambda x: (x == 'above 5').sum())
).reset_index()
result['above_5_percent'] = (result['above_5'] / result['total'] * 100).round(0).astype(int)
result

,embark_town,total,above_5,above_5_percent
0,Cherbourg,168,0,0
1,Queenstown,77,0,0
2,Southampton,644,25,4


9. Заповни відсутні значення віку медіаною.

In [10]:
median_age = np.round(df['age'].median(),2)
df['age'] = df['age'].fillna(median_age)

df

,survived,pclass,sex,age,fare,embarked,class,who,adult_male,deck,embark_town,alive,relatives,relatives_category
0,0,3,male,22.0,7.2500,S,Third,man,True,NaN,Southampton,no,1,1
1,1,1,female,38.0,71.2833,C,First,woman,False,C,Cherbourg,yes,1,1
2,1,3,female,26.0,7.9250,S,Third,woman,False,NaN,Southampton,yes,0,0
3,1,1,female,35.0,53.1000,S,First,woman,False,C,Southampton,yes,1,1
4,0,3,male,35.0,8.0500,S,Third,man,True,NaN,Southampton,no,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,13.0000,S,Second,man,True,NaN,Southampton,no,0,0
887,1,1,female,19.0,30.0000,S,First,woman,False,B,Southampton,yes,0,0
888,0,3,female,28.0,23.4500,S,Third,woman,False,NaN,Southampton,no,3,3
889,1,1,male,26.0,30.0000,C,First,man,True,C,Cherbourg,yes,0,0


10. Створи новий стовпець, де вік представлено категорією, замість числа (наприклад: до 14 років, 14-34 роки, 35-59 років, 60 і більше років). Виконай задачу з використанням користувацької функції. Осіб з невідомим віком познач відповідно.

In [11]:
def age_category(age):
    if pd.isna(age):
        return "unknown"
    elif age < 14:
        return "до 14 років"
    elif age < 35:
        return "14–34 роки"
    elif age < 60:
        return "35–59 років"
    else:
        return "60 і більше років"

df["age_category"] = df["age"].apply(age_category)

df

,survived,pclass,sex,age,fare,embarked,class,who,adult_male,deck,embark_town,alive,relatives,relatives_category,age_category
0,0,3,male,22.0,7.2500,S,Third,man,True,NaN,Southampton,no,1,1,14–34 роки
1,1,1,female,38.0,71.2833,C,First,woman,False,C,Cherbourg,yes,1,1,35–59 років
2,1,3,female,26.0,7.9250,S,Third,woman,False,NaN,Southampton,yes,0,0,14–34 роки
3,1,1,female,35.0,53.1000,S,First,woman,False,C,Southampton,yes,1,1,35–59 років
4,0,3,male,35.0,8.0500,S,Third,man,True,NaN,Southampton,no,0,0,35–59 років
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,13.0000,S,Second,man,True,NaN,Southampton,no,0,0,14–34 роки
887,1,1,female,19.0,30.0000,S,First,woman,False,B,Southampton,yes,0,0,14–34 роки
888,0,3,female,28.0,23.4500,S,Third,woman,False,NaN,Southampton,no,3,3,14–34 роки
889,1,1,male,26.0,30.0000,C,First,man,True,C,Cherbourg,yes,0,0,14–34 роки


11. Перевір, в якій віковій категорії була найвища смертність.     
Для цього рекомендується перетворити стовпець 'alive' в булевий тип.    
 Потім підрахувати загальну кількість пасажирів та кількість тих, хто не вижив.      
 Потім обчисли відносний показниках для кожної категорії.

In [12]:
df['alive'] = df['alive'].map({'yes': True, 'no': False})

result = df.groupby('age_category').agg(
    total=('alive', 'count'),
    not_survived=('alive', lambda x: (~x).sum())
).reset_index()
result['death_rate'] = (result['not_survived'] / result['total'] * 100).round(2)
result.sort_values('death_rate', ascending=False)

,age_category,total,not_survived,death_rate
2,60 і більше років,26,19,73.08
0,14–34 роки,585,379,64.79
1,35–59 років,209,122,58.37
3,до 14 років,71,29,40.85
